### Setting up the Library, API Key, & Stage 2 Folder

In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY = os.environ.get("SECTORS_API_KEY", "")
BASE_URL = "https://api.sectors.app/v2"
HEADERS = {"Authorization": API_KEY}

END_DATE = datetime.now().strftime("%Y-%m-%d")
START_DATE = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d")

CACHE_DIR = "cache"
STAGE2_DIR = os.path.join(CACHE_DIR, "stage2")
SHORTLIST_PATH = os.path.join(CACHE_DIR, "stage2_shortlist.csv")
MANIFEST_PATH = os.path.join(CACHE_DIR, "stage2_manifest.csv")

for sub in ["broker", "foreign", "news", "filings"]:
    os.makedirs(os.path.join(STAGE2_DIR, sub), exist_ok=True)

print(f"Setup complete. Confirmation window: {START_DATE} to {END_DATE}")

Setup complete. Confirmation window: 2026-06-13 to 2026-09-11


### Load Shortlist & Check Credit Estimate

In [2]:
shortlist_df = pd.read_csv(SHORTLIST_PATH)
total_shortlist = len(shortlist_df)

MAX_STOCKS = None  

if MAX_STOCKS:
    target_df = shortlist_df.head(MAX_STOCKS).copy()
else:
    target_df = shortlist_df.copy()

target_symbols = target_df["symbol"].tolist()
est_credit = len(target_symbols) * 5

print(f"Total stocks on shortlist: {total_shortlist} stocks")
print(f"Stocks to be fetched: {len(target_symbols)} stocks")
print(f"Estimated credit requirement: {est_credit} credits (5 credits/stock)")
target_df.head()

Total stocks on shortlist: 63 stocks
Stocks to be fetched: 63 stocks
Estimated credit requirement: 315 credits (5 credits/stock)


,symbol,stage1_score
0,JECC,1.000000
1,MGNA,0.996657
2,MSKY,0.950188
3,VINS,0.857666
4,KLIN,0.821025


### Helper Functions (Manifest & Retry 429)

In [3]:
def load_manifest() -> pd.DataFrame:
    if os.path.exists(MANIFEST_PATH):
        return pd.read_csv(MANIFEST_PATH)
    return pd.DataFrame(columns=[
        "symbol", "broker_status", "foreign_status",
        "news_status", "filings_status", "last_attempt",
    ])

def save_manifest(df: pd.DataFrame) -> None:
    df.to_csv(MANIFEST_PATH, index=False)

def get_row(manifest: pd.DataFrame, symbol: str) -> dict:
    match = manifest[manifest["symbol"] == symbol]
    if match.empty:
        return {
            "symbol": symbol, "broker_status": "pending",
            "foreign_status": "pending", "news_status": "pending",
            "filings_status": "pending", "last_attempt": "",
        }
    return match.iloc[0].to_dict()

def upsert_row(manifest: pd.DataFrame, row: dict) -> pd.DataFrame:
    manifest = manifest[manifest["symbol"] != row["symbol"]]
    return pd.concat([manifest, pd.DataFrame([row])], ignore_index=True)

def request_with_retry(url: str, params: dict, max_retries: int = 5):
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, headers=HEADERS, params=params, timeout=20)
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"[429: wait {wait}s] ", end="", flush=True)
                time.sleep(wait)
                continue
            return None
        except requests.exceptions.RequestException:
            time.sleep(1)
            continue
    return None

print("Helper manifest & retry ready")

Helper manifest & retry ready


### Definition of Fetch Functions by Component (Broker, Foreign, News, Filings)

In [4]:
def fetch_broker(symbol: str) -> bool:
    path = os.path.join(STAGE2_DIR, "broker", f"{symbol}.csv")
    if os.path.exists(path):
        return True
    data = request_with_retry(
        f"{BASE_URL}/broker-summary/{symbol}/top/",
        {"start": START_DATE, "end": END_DATE, "n_brokers": 5},
    )
    if data is None:
        return False
    buyers = data.get("top_buyers", [])
    sellers = data.get("top_sellers", [])
    top_buy_net = buyers[0].get("net_idr", 0) if buyers else 0
    top_sell_net = sellers[0].get("net_idr", 0) if sellers else 0
    pd.DataFrame([{
        "symbol": symbol,
        "broker_net_dominance": top_buy_net + top_sell_net,
        "top_accumulator": buyers[0]["broker_code"] if buyers else None,
        "top_distributor": sellers[0]["broker_code"] if sellers else None,
    }]).to_csv(path, index=False)
    return True

def fetch_foreign(symbol: str) -> bool:
    path = os.path.join(STAGE2_DIR, "foreign", f"{symbol}.csv")
    if os.path.exists(path):
        return True
    data = request_with_retry(
        f"{BASE_URL}/foreign-flow/{symbol}/",
        {"start": START_DATE, "end": END_DATE},
    )
    if data is None:
        return False
    points = data.get("data", [])
    net_total = sum(p.get("net_foreign_inflow", 0) for p in points)
    pd.DataFrame([{
        "symbol": symbol,
        "foreign_net_total": net_total,
        "n_days": len(points),
    }]).to_csv(path, index=False)
    return True

def fetch_news(symbol: str) -> bool:
    path = os.path.join(STAGE2_DIR, "news", f"{symbol}.csv")
    if os.path.exists(path):
        return True
    data = request_with_retry(
        f"{BASE_URL}/news/",
        {"symbols": symbol, "extension": "idx", "start": START_DATE, "end": END_DATE, "limit": 10},
    )
    if data is None:
        return False
    articles = data if isinstance(data, list) else data.get("results", data.get("data", []))
    pd.DataFrame([{
        "symbol": symbol,
        "n_news_articles": len(articles) if articles else 0,
    }]).to_csv(path, index=False)
    return True

def fetch_filings(symbol: str) -> bool:
    path = os.path.join(STAGE2_DIR, "filings", f"{symbol}.csv")
    if os.path.exists(path):
        return True
    data = request_with_retry(
        f"{BASE_URL}/filings/",
        {"symbol": symbol, "start": START_DATE, "end": END_DATE, "limit": 10},
    )
    if data is None:
        return False
    filings = data if isinstance(data, list) else data.get("results", data.get("data", []))
    pd.DataFrame([{
        "symbol": symbol,
        "n_filings": len(filings) if filings else 0,
    }]).to_csv(path, index=False)
    return True

print("Signal component function 4 ready")

Signal component function 4 ready


### Fetch Stage 2 Execution (Loop with Auto-Resume)

In [5]:
manifest = load_manifest()

print(f"Fetching Stage 2 signals for {len(target_symbols)} shortlisted stocks")
print(f"Period: {START_DATE} to {END_DATE}\n")

for i, symbol in enumerate(target_symbols):
    row = get_row(manifest, symbol)

    # Broker Dominance
    if row["broker_status"] != "done":
        row["broker_status"] = "done" if fetch_broker(symbol) else "failed"
        time.sleep(0.5)

    # Foreign Flow
    if row["foreign_status"] != "done":
        row["foreign_status"] = "done" if fetch_foreign(symbol) else "failed"
        time.sleep(0.5)

    # News
    if row["news_status"] != "done":
        row["news_status"] = "done" if fetch_news(symbol) else "failed"
        time.sleep(0.5)

    # Filings
    if row["filings_status"] != "done":
        row["filings_status"] = "done" if fetch_filings(symbol) else "failed"
        time.sleep(0.5)

    row["last_attempt"] = str(pd.Timestamp.now())
    manifest = upsert_row(manifest, row)
    save_manifest(manifest)

    statuses = f"B:{row['broker_status'][:1]} F:{row['foreign_status'][:1]} N:{row['news_status'][:1]} Fi:{row['filings_status'][:1]}"
    print(f"  [{i + 1:3d}/{len(target_symbols)}] {symbol:<8} {statuses}")

print("\nFetch Stage 2 complete")

Fetching Stage 2 signals for 63 shortlisted stocks
Period: 2026-06-13 to 2026-09-11

  [  1/63] JECC     B:d F:d N:d Fi:d
  [  2/63] MGNA     B:d F:d N:d Fi:d
  [  3/63] MSKY     B:d F:d N:d Fi:d
  [  4/63] VINS     B:d F:d N:d Fi:d
  [  5/63] KLIN     B:d F:d N:d Fi:d
  [  6/63] IKBI     B:d F:d N:d Fi:d
  [  7/63] SAPX     B:d F:d N:d Fi:d
  [  8/63] PLAN     B:d F:d N:d Fi:d
  [  9/63] IDEA     B:d F:d N:d Fi:d
  [ 10/63] SEMA     B:d F:d N:d Fi:d
  [ 11/63] JELI     B:d F:d N:d Fi:d
  [ 12/63] LPLI     B:d F:d N:d Fi:d
  [ 13/63] ESTI     B:d F:d N:d Fi:d
  [ 14/63] NASA     B:d F:d N:d Fi:d
  [ 15/63] KBLM     B:d F:d N:d Fi:d
  [ 16/63] OLIV     B:d F:d N:d Fi:d
  [ 17/63] WOMF     B:d F:d N:d Fi:d
  [ 18/63] CCSI     B:d F:d N:d Fi:d
  [ 19/63] BPTR     B:d F:d N:d Fi:d
  [ 20/63] MANG     B:d F:d N:d Fi:d
  [ 21/63] AYLS     B:d F:d N:d Fi:d
  [ 22/63] GHON     B:d F:d N:d Fi:d
  [ 23/63] WIRG     B:d F:d N:d Fi:d
  [ 24/63] RODA     B:d F:d N:d Fi:d
  [ 25/63] FOLK     B:d F:d

### Manifest Summary and Signal Compilation

In [6]:
manifest = load_manifest()
print("Fetch Status Summary")
for col in ["broker_status", "foreign_status", "news_status", "filings_status"]:
    print(f"{col:<16}: {manifest[col].value_counts().to_dict()}")

compiled_rows = []
for symbol in target_symbols:
    row_data = {"symbol": symbol}
    
    # Read brokers
    p_b = os.path.join(STAGE2_DIR, "broker", f"{symbol}.csv")
    if os.path.exists(p_b):
        df_b = pd.read_csv(p_b)
        row_data.update(df_b.iloc[0].to_dict())
        
    # Read foreign
    p_f = os.path.join(STAGE2_DIR, "foreign", f"{symbol}.csv")
    if os.path.exists(p_f):
        df_f = pd.read_csv(p_f)
        row_data.update(df_f.iloc[0].to_dict())

    # Read news
    p_n = os.path.join(STAGE2_DIR, "news", f"{symbol}.csv")
    if os.path.exists(p_n):
        df_n = pd.read_csv(p_n)
        row_data.update(df_n.iloc[0].to_dict())

    # Read filings
    p_fi = os.path.join(STAGE2_DIR, "filings", f"{symbol}.csv")
    if os.path.exists(p_fi):
        df_fi = pd.read_csv(p_fi)
        row_data.update(df_fi.iloc[0].to_dict())
        
    compiled_rows.append(row_data)

stage2_summary = pd.DataFrame(compiled_rows)
stage2_summary.to_csv(os.path.join(CACHE_DIR, "stage2_signals_summary.csv"), index=False)

print(f"\nSignal data successfully compiled: {len(stage2_summary)} stocks saved")
stage2_summary.head(10)

Fetch Status Summary
broker_status   : {'done': 63}
foreign_status  : {'done': 63}
news_status     : {'done': 63}
filings_status  : {'done': 63}

Signal data successfully compiled: 63 stocks saved


,symbol,broker_net_dominance,top_accumulator,top_distributor,foreign_net_total,n_days,n_news_articles,n_filings
0,JECC,3350000,RB,OK,132787500,61,1,0
1,MGNA,-656483900,MG,XA,-3253452400,61,2,0
2,MSKY,2616401700,XL,CC,-1905563900,61,1,0
3,VINS,10071100,XL,XC,-27001000,61,0,0
4,KLIN,102241300,OD,KK,-131507700,61,0,0
5,IKBI,-36515000,SQ,EP,-132489300,61,9,2
6,SAPX,2731288300,MG,XL,-1094381800,61,2,0
7,PLAN,15240600,CP,XL,781103500,61,0,0
8,IDEA,143724600,AZ,LG,-901706800,61,4,0
9,SEMA,344787700,CC,FS,-363531500,61,0,0
